# Notebook 0: Setup & Configuration

Welcome to the Prompt Engineering for Mistral on EKS tutorial series!

## What You'll Learn in This Series

This tutorial will teach you how to craft effective prompts for Mistral models. You'll learn:

- How to structure prompts with system and user messages
- Techniques for defining roles and purposes
- How to organize complex instructions clearly
- Using delimiters and formatting for safety and clarity
- Few-shot prompting to guide model behavior
- Controlling output formats for reliable parsing
- Step-by-step reasoning for complex tasks
- Reducing hallucinations and grounding responses

## Prerequisites

- A running Mistral model endpoint on EKS (vLLM with OpenAI-compatible API)
- Python 3.8+
- Basic Python knowledge

## Reference

- [Mistral Prompting Documentation](https://docs.mistral.ai/guides/prompting/)

---
## Section 1: Install Dependencies

In [ ]:
%pip install openai --quiet

---
## Section 2: Configuration

### Endpoint Configuration

Set the endpoint URL for your deployed Mistral model on EKS.

**How to get your endpoint URL:**
1. If using port-forwarding: `http://localhost:8000/v1`
2. If using LoadBalancer service: `http://<load-balancer-url>:8000/v1`
3. If using Ingress: `http://<ingress-url>/v1`

In [ ]:
# =============================================================================
# CONFIGURATION - Modify these settings for your deployment
# =============================================================================

import os

# Endpoint URL - set this to your deployed model's URL
# Can also be set via environment variable: MISTRAL_ENDPOINT_URL
ENDPOINT_URL = os.getenv("MISTRAL_ENDPOINT_URL", "http://localhost:8000/v1")

# Model name - must match the model name in your vLLM deployment
# Can also be set via environment variable: MISTRAL_MODEL_NAME  
MODEL_NAME = os.getenv("MISTRAL_MODEL_NAME", "mistralai/Ministral-8B-Instruct-2410")

# Default inference parameters
DEFAULT_MAX_TOKENS = 1024
DEFAULT_TEMPERATURE = 0.5

print(f"Endpoint URL: {ENDPOINT_URL}")
print(f"Model Name: {MODEL_NAME}")

---
## Section 3: Initialize the OpenAI Client

vLLM exposes an OpenAI-compatible API, so we use the OpenAI Python SDK to connect to it.

In [ ]:
from openai import OpenAI

# Create the OpenAI client pointing to our vLLM endpoint
client = OpenAI(
    base_url=ENDPOINT_URL,
    api_key="not-needed"  # vLLM doesn't require an API key, but the SDK requires this field
)

print(f"OpenAI client initialized for endpoint: {ENDPOINT_URL}")

---
## Section 4: Helper Functions

These helper functions wrap the OpenAI API calls. You'll use them throughout all notebooks.

In [ ]:
def call_mistral(
    user_prompt: str,
    system_prompt: str = None,
    model: str = MODEL_NAME,
    max_tokens: int = DEFAULT_MAX_TOKENS,
    temperature: float = DEFAULT_TEMPERATURE
) -> str:
    """
    Call a Mistral model on EKS using the OpenAI-compatible API.

    Args:
        user_prompt: The user message to send
        system_prompt: Optional system prompt to set model behavior
        model: The model name (must match vLLM deployment)
        max_tokens: Maximum tokens in the response
        temperature: Sampling temperature (0-1)

    Returns:
        The model's response text
    """
    # Build messages list
    messages = []
    
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    
    messages.append({"role": "user", "content": user_prompt})

    try:
        # Call the model using OpenAI-compatible API
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature
        )
        
        return response.choices[0].message.content
    
    except Exception as e:
        return f"[Error calling model: {e}]"


def call_mistral_with_messages(
    messages: list,
    system_prompt: str = None,
    model: str = MODEL_NAME,
    max_tokens: int = DEFAULT_MAX_TOKENS,
    temperature: float = DEFAULT_TEMPERATURE
) -> str:
    """
    Call a Mistral model with a full messages list (for few-shot prompting).

    Args:
        messages: List of message dicts with 'role' and 'content'
                  Supports 'system', 'user', and 'assistant' roles.
        system_prompt: Optional additional system prompt (prepended to messages)
        model: The model name (must match vLLM deployment)
        max_tokens: Maximum tokens in the response
        temperature: Sampling temperature (0-1)

    Returns:
        The model's response text
    """
    # Build the final messages list
    final_messages = []
    
    # Add system prompt if provided separately
    if system_prompt:
        final_messages.append({"role": "system", "content": system_prompt})
    
    # Add all provided messages
    final_messages.extend(messages)

    try:
        # Call the model using OpenAI-compatible API
        response = client.chat.completions.create(
            model=model,
            messages=final_messages,
            max_tokens=max_tokens,
            temperature=temperature
        )
        
        return response.choices[0].message.content
    
    except Exception as e:
        return f"[Error calling model: {e}]"


print("Helper functions defined successfully!")

---
## Section 5: Test Your Setup

Let's verify everything works with a simple test.

In [ ]:
# Simple test - the model should respond with a greeting
response = call_mistral(
    user_prompt="Respond with exactly one word: Hello",
    temperature=0.0  # Use 0 for deterministic output
)

print(f"Endpoint: {ENDPOINT_URL}")
print(f"Model: {MODEL_NAME}")
print(f"Response: {response}")
print("\n" + "="*50)

if "Error" not in response:
    print("\n✅ Setup complete! You're ready to start the tutorial.")
else:
    print("\n❌ Setup failed. Please check:")
    print("   1. Is your model endpoint running?")
    print("   2. Is the ENDPOINT_URL correct?")
    print("   3. Is the MODEL_NAME correct?")

---
## Section 6: Understanding the Response Structure

Let's look at what comes back from the API in more detail.

In [ ]:
# Get the full response object
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
    max_tokens=100,
    temperature=0.0
)

print("Full API response structure:")
print(f"  ID: {response.id}")
print(f"  Model: {response.model}")
print(f"  Created: {response.created}")
print(f"  Choices: {len(response.choices)}")
print(f"  Message Role: {response.choices[0].message.role}")
print(f"  Message Content: {response.choices[0].message.content}")
print(f"  Finish Reason: {response.choices[0].finish_reason}")
if response.usage:
    print(f"  Usage - Prompt Tokens: {response.usage.prompt_tokens}")
    print(f"  Usage - Completion Tokens: {response.usage.completion_tokens}")
    print(f"  Usage - Total Tokens: {response.usage.total_tokens}")

The OpenAI-compatible API response includes:
- `choices[0].message.content`: The model's response text
- `choices[0].message.role`: Will be "assistant"
- `choices[0].finish_reason`: Why generation stopped (e.g., "stop", "length")
- `usage`: Token counts (prompt_tokens, completion_tokens, total_tokens)

---
## Exercise: Adjust Temperature

Temperature controls randomness. Lower = more deterministic, higher = more creative.

Run the same prompt multiple times with different temperatures to see the effect.

In [ ]:
prompt = "Write a one-sentence description of a cat."

print("=" * 50)
print("Temperature: 0.0 (deterministic)")
print("=" * 50)
for i in range(3):
    response = call_mistral(prompt, temperature=0.0)
    print(f"Run {i+1}: {response}")

print("\n" + "=" * 50)
print("Temperature: 1.0 (creative)")
print("=" * 50)
for i in range(3):
    response = call_mistral(prompt, temperature=1.0)
    print(f"Run {i+1}: {response}")

---
## Next Steps

Your setup is complete! Proceed to **Notebook 1: Basic Prompt Structure** to start learning prompt engineering techniques.

📚 [Continue to Notebook 1 →](01_basic_prompt_structure.ipynb)